In [34]:
import sys

sys.path.insert(0, "/workspaces/SolvMate")
import pandas as pd
import numpy as np
import os
import logging
from typing import Tuple, Dict, List, Optional
from dotenv import load_dotenv
from supabase import create_client, Client
from src.utils.input import get_dataframe, run_Import
from src.utils.get_Value import get_value, get_valueExt
from pathlib import Path

class CurRisk:
    
    def __init__(self):

        self.__inputColumns = None
        self.__localCurrency = None
        self.__lacTPAbsValShockUp = 0.0
        self.__lacTPAbsValShockDown = 0.0
        self.__lacTPRelGrossShockUp = 0.0
        self.__lacTPRelGrossShockDown = 0.0
        self.__assetFlag = None
        self.__liabFlag = None
        self.__currencyShocks = None
        self.__currrencyShockStd = 0.25
        self.output = pd.DataFrame()
        self.aggOutput = pd.DataFrame(columns=['Shock','Assets', 'Liabilities', 'AssetsShocked', 'LiabilitiesShockedNet', 'LiabilitiesShockedGross'])

    def __readInput(self, input_file):
        
        # Initialize Supabase client
        load_dotenv()
        supabase: Client = create_client(
            os.environ.get("SUPABASE_URL"),
            os.environ.get("SUPABASE_KEY")
        )

        # Load input dataframe of input columns
        data_id_enriched = run_Import(input_file, ["CurrR"])
        data_id_enriched_Basic = run_Import(input_file, ["Basic input"])
        inputColumns = get_dataframe(data_id_enriched, "MARKET_FX")

        # For further calculations, empty fields get filled with a Null instead of carrying care of empty fields
        # The required columns of Input get handled in a global Error-Concept to treat it with an Error message.
        # It will not get checked in this file.

        pd.set_option('future.no_silent_downcasting', True)
        inputColumns = inputColumns.replace("", np.nan)
        inputColumns = inputColumns.fillna(0)
        inputColumns = inputColumns.infer_objects(copy=False)
        cols_to_convert = [col for col in inputColumns.columns if col != 'FX_FOREIGN_CCY_']
        try:
            inputColumns[cols_to_convert] = inputColumns[cols_to_convert].astype(float)
        except Exception as e:
            raise ValueError(f"Conversion of inputColumns to float failed: {e}")
        self.__inputColumns = inputColumns

        #Load simple input data 
        self.__localCurrency = get_value("FX_LOCAL_CCY", data_id_enriched) 
        if self.__localCurrency is None:
            raise ValueError("Local currency not found in the CurrR sheet.")

        self.__lacTPAbsValShockUp = get_valueExt("FX_UP_LAC", data_id_enriched)
        self.__lacTPAbsValShockDown = get_valueExt("FX_DOWN_LAC", data_id_enriched)
        self.__lacTPRelGrossShockUp = get_valueExt("FX_UP_LAC_REL", data_id_enriched)
        self.__lacTPRelGrossShockDown = get_valueExt("FX_DN_LAC_REL", data_id_enriched)

        # Input Method
        inputMethod = get_value("INFO_CURR_ASSETS_MANUAL_INPUT", data_id_enriched_Basic)
        if inputMethod is None:
            raise ValueError("Error: Input Method for currency risk not found in the Basic input sheet.")
        start_index = inputMethod.find(" ", 0)
        end_index = inputMethod.find("-", start_index)
        self.__assetFlag = inputMethod[start_index+1:end_index-1]
        start_index = inputMethod.find("Liab: ", 0)
        self.__liabFlag = inputMethod[start_index+6:]

        print("assetFlag:" + self.__assetFlag + " liabFlag:" + self.__liabFlag)

        if not self.__assetFlag:  
            raise ValueError("Error: assetFlag must not be empty or unfilled. Check inputMethod for Currency Risk.")
        if not self.__liabFlag:  # Überprüft, ob assetFlag leer oder None ist
            raise ValueError("Error: liabFlag must not be empty or unfilled. Check inputMethod for Currency Risk.")

        # Load currency shock percentage values
        currencyShocks_response = supabase.table('currency_shock').select('*').execute()
        self.__currencyShocks = pd.DataFrame(currencyShocks_response.data)
        if self.__currencyShocks.empty:
            raise ValueError("Error: currency shock table of percentages could not be loaded or is empty.")
    
    #Main Function to calculate the market risk for foreign exchange
    def calculate(self,input_file):

        self.__readInput(input_file)
        self.__copyInputColumns()
        self.__calcRiskFactor()
        self.__calcGrossValues()
        self.__calcNetValues()
        self.__identifyRelevantShocks()
        self.__calcAggOutput()

    def __copyInputColumns(self):

        self.output["ForeignCurrency"] = self.__inputColumns["FX_FOREIGN_CCY_"]
       
        #For local currency there is no shock and values have to be set to 0
        for idx, row in self.output.iterrows():
            if row["ForeignCurrency"] == self.__localCurrency:
                self.output.at[idx, "Asset_Input"] = 0.0
                self.output.at[idx, "Liab_Input"] = 0.0
                self.output.at[idx, "AssetShockUp_Input"] = 0.0
                self.output.at[idx, "LiabShockUpAfterLACTP_Input"] = 0.0
                self.output.at[idx, "LiabShockUp_Input"] = 0.0
                self.output.at[idx, "AssetShockDown_Input"] = 0.0
                self.output.at[idx, "LiabShockDown_Input"] = 0.0
                self.output.at[idx, "LiabShockDownAfterLACTP_Input"] = 0.0
            else:
                self.output.at[idx, "Asset_Input"] = self.__inputColumns.at[idx, "FX_A_BC_"]
                self.output.at[idx, "Liab_Input"] = self.__inputColumns.at[idx, "FX_L_BC_"]
                self.output.at[idx, "AssetShockUp_Input"] = self.__inputColumns.at[idx, "FX_A_UP_"]
                self.output.at[idx, "LiabShockUpAfterLACTP_Input"] = self.__inputColumns.at[idx, "FX_L_UP_A_"]
                self.output.at[idx, "LiabShockUp_Input"] = self.__inputColumns.at[idx, "FX_L_UP_B_"]
                self.output.at[idx, "AssetShockDown_Input"] = self.__inputColumns.at[idx, "FX_A_DN_"]
                self.output.at[idx, "LiabShockDown_Input"] = self.__inputColumns.at[idx, "FX_L_DN_B_"]
                self.output.at[idx, "LiabShockDownAfterLACTP_Input"] = self.__inputColumns.at[idx, "FX_L_DN_A_"]
    
    def __calcRiskFactor(self):
        
        # Boolean-Spalte erzeugen: True, wenn Wert in den Spaltennamen von currencyShocks vorkommt
        self.output["ForeignCurrencyPeggedEUR"] = self.output["ForeignCurrency"].apply(
            lambda x: x in self.__currencyShocks.columns
        )
    
        self.output["RiskFactor"] = 0.0
        # Copy value from currencyShocks, if ForeignCurrencyPeggedEUR True is true
        for idx, row in self.output.iterrows():
            if row["ForeignCurrencyPeggedEUR"]:
                self.output.at[idx, "RiskFactor"] = self.__currencyShocks[row["ForeignCurrency"]].iloc[0]
            else:
                self.output.at[idx, "RiskFactor"] = self.__currrencyShockStd
                
    def __calcGrossValues(self):
        
        for idx, row in self.output.iterrows():
            if row["ForeignCurrency"] == self.__localCurrency:
                self.output.at[idx, "AssetSCRGross"] = 0.0
                self.output.at[idx, "LiabSCRGross"] = 0.0
            else:
                #Multiply Assets and Liabilities with RiskFactor
                self.output.at[idx, "AssetSCRGross"] = self.output.at[idx, "RiskFactor"] * self.output.at[idx, "Asset_Input"]
                self.output.at[idx, "LiabSCRGross"] = self.output.at[idx, "RiskFactor"] * self.output.at[idx, "Liab_Input"]
            
            if self.__assetFlag == "Manual BC+SH":
                self.output.at[idx, "AssetShockDown"] =self.output.at[idx, "AssetShockDown_Input"]
                self.output.at[idx, "AssetShockUp"] =self.output.at[idx, "AssetShockUp_Input"]
            else:
                self.output.at[idx, "AssetShockDown"] = self.output.at[idx, "Asset_Input"] - self.output.at[idx, "AssetSCRGross"]
                self.output.at[idx, "AssetShockUp"] = self.output.at[idx, "Asset_Input"] + self.output.at[idx, "AssetSCRGross"]
            
            if self.__liabFlag == "Manual BC+SH":
                self.output.at[idx, "LiabDownGross"] =self.output.at[idx, "LiabShockDown_Input"]
                self.output.at[idx, "LiabUpGross"] =self.output.at[idx, "LiabShockUp_Input"]
            else:
                self.output.at[idx, "LiabDownGross"] = self.output.at[idx, "Liab_Input"] - self.output.at[idx, "LiabSCRGross"]
                self.output.at[idx, "LiabUpGross"] = self.output.at[idx, "Liab_Input"] + self.output.at[idx, "LiabSCRGross"]
                
            #SCR = Max (0, (Asset Basis -Liabilities Basis)-(Asset Shocked -Liabilities Shocked))
            self.output.at[idx, "SCRDownGross"] = max(0,(self.output.at[idx, "Asset_Input"] -self.output.at[idx, "Liab_Input"])-(self.output.at[idx, "AssetShockDown"]-self.output.at[idx, "LiabDownGross"]))
            self.output.at[idx, "SCRUpGross"] = max(0,(self.output.at[idx, "Asset_Input"] -self.output.at[idx, "Liab_Input"])-(self.output.at[idx, "AssetShockUp"]-self.output.at[idx, "LiabUpGross"]))


    def __calcNetValues(self):
        
        #Calculate net values 
        #Net value = Gross value - Part of Lac TP
        for idx, row in self.output.iterrows():
            if self.__liabFlag == "Manual BC+SH":
                    self.output.at[idx, "LiabDownNet"] =self.output.at[idx, "LiabShockDownAfterLACTP_Input"]
                    self.output.at[idx, "LiabUpNet"] =self.output.at[idx, "LiabShockUpAfterLACTP_Input"]
            else:
                if self.output['SCRDownGross'].sum() == 0:
                    self.output.at[idx, "LiabDownNet"] = self.output.at[idx, "LiabDownGross"]
                else:
                    self.output.at[idx, "LiabDownNet"] = self.output.at[idx, "LiabDownGross"]- self.lacTPAbsValShockDown*self.output.at[idx, "SCRDown-Gross"]/self.output['SCRDownGross'].sum()-self.lacTPRelGrossShockDown*self.output.at[idx, "SCRDownGross"]
                    
                if self.output['SCRUpGross'].sum() == 0:
                    self.output.at[idx, "LiabUpNet"] = self.output.at[idx, "LiabUpGross"]
                else:
                    self.output.at[idx, "LiabUpNet"] = self.output.at[idx, "LiabUpGross"]-self.lacTPAbsValShockUp*self.output.at[idx, "SCRUpGross"]/self.output['SCRUpGross'].sum()-self.lacTPRelGrossShockUp*self.output.at[idx, "SCRUpGross"]
            
            #SCR = Max (0, (Asset Basis -Liabilities Basis)-(Asset Shocked -Liabilities Shocked Net))
            self.output.at[idx, "SCRDownNet"] = max(0,(self.output.at[idx, "Asset_Input"] -self.output.at[idx, "Liab_Input"])-(self.output.at[idx, "AssetShockDown"]-self.output.at[idx, "LiabDownNet"]))
            self.output.at[idx, "SCRUpNet"] = max(0,(self.output.at[idx, "Asset_Input"] -self.output.at[idx, "Liab_Input"])-(self.output.at[idx, "AssetShockUp"]-self.output.at[idx, "LiabUpNet"]))
        
        
    def __identifyRelevantShocks(self):

        #Find relevant shocks (Up or Down)
        for idx, row in self.output.iterrows():
            if self.output.at[idx, "SCRDownNet"] == self.output.at[idx, "SCRUpNet"]:
                if self.output.at[idx, "SCRUpGross"] < self.output.at[idx, "SCRDownGross"]:
                    self.output.at[idx, "RelevantShock"]= "Downward"
                else:
                    self.output.at[idx, "RelevantShock"]= "Upward"
            else:
                if self.output.at[idx, "SCRUpNet"] < self.output.at[idx, "SCRDownNet"]:
                    self.output.at[idx, "RelevantShock"]= "Downward"
                else:
                    self.output.at[idx, "RelevantShock"]= "Upward"

    def __calcAggOutput(self):

        #Aggregate values selected by RelevantShock
        filtered_output = self.output[self.output['RelevantShock'] == 'Upward']

        self.aggOutput.loc[1, 'Shock'] = "Upward"
        self.aggOutput.loc[1, 'Assets'] = filtered_output["Asset_Input"].sum()
        self.aggOutput.loc[1, 'Liabilities'] = filtered_output["Liab_Input"].sum()
        self.aggOutput.loc[1, 'AssetsShocked'] = filtered_output['AssetShockUp'].sum()
        self.aggOutput.loc[1, 'LiabilitiesShockedNet'] = filtered_output['LiabUpNet'].sum()
        self.aggOutput.loc[1, 'SCRNet'] = filtered_output['SCRUpNet'].sum()
        self.aggOutput.loc[1, 'LiabilitiesShockedGross'] = filtered_output['LiabUpGross'].sum()
        self.aggOutput.loc[1, 'SCRGross'] = filtered_output['SCRUpGross'].sum()

        filtered_output = self.output[self.output['RelevantShock'] == 'Downward']

        self.aggOutput.loc[2, 'Shock'] = "Downward"
        self.aggOutput.loc[2, 'Assets'] = filtered_output["Asset_Input"].sum()
        self.aggOutput.loc[2, 'Liabilities'] = filtered_output["Liab_Input"].sum()
        self.aggOutput.loc[2, 'AssetsShocked'] = filtered_output['AssetShockDown'].sum()
        self.aggOutput.loc[2, 'LiabilitiesShockedNet'] = filtered_output['LiabDownNet'].sum()
        self.aggOutput.loc[2, 'SCRNet'] = filtered_output['SCRDownNet'].sum()
        self.aggOutput.loc[2, 'LiabilitiesShockedGross'] = filtered_output['LiabDownGross'].sum()
        self.aggOutput.loc[2, 'SCRGross'] = filtered_output['SCRDownGross'].sum()      


In [33]:
input_file = Path("/workspaces/SolvMate/input/02.10_SAS_Input_CurrR.xls")

curRisk = CurRisk()
curRisk.calculate(input_file)

curRisk.output.to_excel("/workspaces/SolvMate/outputs/Output_CurrR.xlsx", index=False)
curRisk.aggOutput.to_excel("/workspaces/SolvMate/outputs/AggOutput_CurrR.xlsx", index=False)
curRisk.aggOutput

2025-07-16 15:57:53,392 - httpx - INFO - HTTP Request: GET https://hgdgxwrwohwkmywruxat.supabase.co/rest/v1/data_id?select=%2A&WORKSHEET=eq.CurrR "HTTP/2 200 OK"
2025-07-16 15:57:53,979 - httpx - INFO - HTTP Request: GET https://hgdgxwrwohwkmywruxat.supabase.co/rest/v1/data_id?select=%2A&WORKSHEET=eq.Basic+input "HTTP/2 200 OK"
2025-07-16 15:57:54,563 - httpx - INFO - HTTP Request: GET https://hgdgxwrwohwkmywruxat.supabase.co/rest/v1/currency_shock?select=%2A "HTTP/2 200 OK"


Assets: Manual BC+SH - Liab: Manual BC+SH Manual BC+SH Manual BC+SH


,Shock,Assets,Liabilities,AssetsShocked,LiabilitiesShockedNet,LiabilitiesShockedGross,SCRNet,SCRGross
1,Upward,14819.663213,12750.13726,14819.663213,15255.702688,12750.13726,4134.673303,0.0
2,Downward,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
